In [47]:
from fpl_project.scripts import config
from fpl_project.scripts.model_utils.data import *
from fpl_project.scripts.model_utils.trainer import *
from fpl_project.scripts.data_utils.features_processing import *

In [48]:
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.nn import HuberLoss, MSELoss
from torch.utils.data import DataLoader, Dataset
from torchinfo import summary

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

In [49]:
from datetime import datetime

In [50]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment(f"FPL-Points-Predictions_{datetime.now()}")

set_seed(77)


2026/05/31 17:47:38 INFO mlflow.tracking.fluent: Experiment with name 'FPL-Points-Predictions_2026-05-31 17:47:36.120622' does not exist. Creating a new experiment.


In [51]:
data = pd.read_parquet(config.FPL_DATA_PATH)
data_seq = pd.read_parquet(config.TIDY_DATA_PATH)

data = sort_data(data)

print(f"Shape: {data.shape}  |  Seasons: {sorted(data.season.unique())}")
print(f"NaN count: {data.isna().sum().sum()}")


print(f"Shape: {data_seq.shape}  |  Seasons: {sorted(data_seq.season.unique())}")
print(f"NaN count: {data_seq.isna().sum().sum()}")


Shape: (105866, 262)  |  Seasons: [np.float64(2223.0), np.float64(2324.0), np.float64(2425.0), np.float64(2526.0)]
NaN count: 0
Shape: (105866, 48)  |  Seasons: [np.int64(2223), np.int64(2324), np.int64(2425), np.int64(2526)]
NaN count: 0


In [52]:
cols_map   = FEATURES_GROUP
device     = "cuda" if torch.cuda.is_available() else "cpu"

SEQ_LEN = 4

BATCH_SIZE = 512
LR         = .001
WEIGHT_DECAY = 0.05
NUM_EPOCHS   = 500

print(f"Device: {device}")


Device: cpu


## Podział danych (chronologiczny — brak leakage między sezonami)
- **Train**: sezony 22/23, 23/24  
- **Valid**: sezon 24/25  
- **Test**:  sezon 25/26


In [53]:
data_splits = train_test_split(data)
data_splits_seq = train_test_split(data_seq)


In [54]:
target_scaler = StandardScaler()

y_train_vals = data_splits["y_train"].values.reshape(-1, 1)
target_scaler.fit(y_train_vals)

for splits_dict in [data_splits, data_splits_seq]:
    for split in ["y_train", "y_valid", "y_test"]:
        vals = splits_dict[split].values.reshape(-1, 1)
        transformed = target_scaler.transform(vals)

        splits_dict[split] = pd.DataFrame(
            transformed,
            index=splits_dict[split].index,
            columns=["total_points"]
        )

print("Rozmiary zbiorów MLP:")
for k, v in data_splits.items():
    print(f"  {k}: {v.shape}")

print("\nRozmiary zbiorów Sekwencyjne:")
for k, v in data_splits_seq.items():
    print(f"  {k}: {v.shape}")

Rozmiary zbiorów MLP:
  X_train: (56230, 261)
  y_train: (56230, 1)
  X_valid: (27283, 261)
  y_valid: (27283, 1)
  X_test: (22353, 261)
  y_test: (22353, 1)

Rozmiary zbiorów Sekwencyjne:
  X_train: (56230, 47)
  y_train: (56230, 1)
  X_valid: (27283, 47)
  y_valid: (27283, 1)
  X_test: (22353, 47)
  y_test: (22353, 1)


In [55]:
num_cols_mlp = (
    [c for c in data.columns if "ema" in c or "lagg" in c]
    + [c for c in cols_map["pre_game_cols"]
       if c in data.columns
       and c not in ["name", "position", "element", "opponent_team",
                     "gw", "code", "season", "kickoff_time"]]
)
num_cols_mlp = list(dict.fromkeys(num_cols_mlp))

cat_cols_mlp = [
    c for c in cols_map["static_cols"]
    if c in data.columns and c not in ["web_name", "player_id", "gw"]
]

print(f"Cechy MLP -> Numeryczne: {len(num_cols_mlp)} | Kategoryczne: {len(cat_cols_mlp)}\n")

fpl_pipe_mlp = FPLDataPipe(
    num_cols=num_cols_mlp, cat_cols=cat_cols_mlp, batch_size=BATCH_SIZE
)
fpl_pipe_mlp.prepare_data(**data_splits)
train_dl_mlp, valid_dl_mlp, test_dl_mlp = fpl_pipe_mlp.get_dataloaders()

x_mlp, y_mlp = next(iter(train_dl_mlp))
INPUT_SIZE_MLP = x_mlp.shape[1]


Cechy MLP -> Numeryczne: 157 | Kategoryczne: 2



In [56]:
seq_feature_cols = [
    c for c in cols_map["fpl_cols"] + cols_map["perf_cols"]
    if c in data_seq.columns and c != "total_points"   # Zmieniono data na data_seq
]
seq_feature_cols = list(dict.fromkeys(seq_feature_cols))

print(f"Cechy Sequence -> Łącznie: {len(seq_feature_cols)}\n")

seq_pipe_conv = SequenceDataPipe(
    feature_cols=seq_feature_cols, seq_len=SEQ_LEN, batch_size=BATCH_SIZE, model_type="conv1d"
)
seq_pipe_conv.prepare_data(**data_splits_seq)
train_dl_conv, valid_dl_conv, test_dl_conv = seq_pipe_conv.get_dataloaders()

seq_pipe_lstm = SequenceDataPipe(
    feature_cols=seq_feature_cols, seq_len=SEQ_LEN, batch_size=BATCH_SIZE, model_type="lstm"
)
seq_pipe_lstm.prepare_data(**data_splits_seq)
train_dl_lstm, valid_dl_lstm, test_dl_lstm = seq_pipe_lstm.get_dataloaders()

Cechy Sequence -> Łącznie: 32

[SequenceDataPipe/conv1d]  train=49697  valid=24156  test=19079  seq_len=4
[SequenceDataPipe/lstm]  train=49697  valid=24156  test=19079  seq_len=4


In [57]:
N_FEATURES_SEQ = len(seq_feature_cols)

In [58]:
class MLP(nn.Module):
    def __init__(self, input_size: int, hidden_sizes: list[int], output_size: int = 1,
                 dropout_rates: list[float] | None = None):
        super().__init__()
        if dropout_rates is None:
            dropout_rates = [0.2] + [0.1] * (len(hidden_sizes) - 1)

        layers = []
        in_dim = input_size
        for h, dr in zip(hidden_sizes, dropout_rates):
            layers += [nn.Linear(in_dim, h), nn.LayerNorm(h), nn.LeakyReLU(0.01), nn.Dropout(dr)]
            in_dim = h
        layers.append(nn.Linear(in_dim, output_size))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, features)
        return self.net(x)


mlp = MLP(input_size=INPUT_SIZE_MLP, hidden_sizes=[64, 32]).to(device)
summary(mlp, input_size=(BATCH_SIZE, INPUT_SIZE_MLP),
        col_names=["input_size", "output_size", "num_params", "trainable"], col_width=18)


Layer (type:depth-idx)                   Input Shape        Output Shape       Param #            Trainable
MLP                                      [512, 161]         [512, 1]           --                 True
├─Sequential: 1-1                        [512, 161]         [512, 1]           --                 True
│    └─Linear: 2-1                       [512, 161]         [512, 64]          10,368             True
│    └─LayerNorm: 2-2                    [512, 64]          [512, 64]          128                True
│    └─LeakyReLU: 2-3                    [512, 64]          [512, 64]          --                 --
│    └─Dropout: 2-4                      [512, 64]          [512, 64]          --                 --
│    └─Linear: 2-5                       [512, 64]          [512, 32]          2,080              True
│    └─LayerNorm: 2-6                    [512, 32]          [512, 32]          64                 True
│    └─LeakyReLU: 2-7                    [512, 32]          [512, 32]   

---
## 2. Conv1D & LSTM — pipeline sekwencyjny

**Strategia cech dla modeli sekwencyjnych**:  
Conv1D i LSTM dostają **surowe (oryginalne, niescalowane) wartości cech** ułożone w okna czasowe.  
- Nie potrzebują ręcznie tworzonych lagów ani EMA — sieć sama uczy się wzorców ze sekwencji.  
- **Brak data leakage**: okno o długości `seq_len` kończy się na **kolejce `t-1`**. Przewidujemy kolejkę `t`.  
- Do wejścia modeli trafiają cechy z grup `fpl_cols` + `perf_cols` (metryki wydajnościowe i FPL) — bez kolumn statycznych (position/team) które są niezmienne w czasie; te są doklejane osobno jako metadane lub pomijane.

Kształty:
- **Conv1D**: `(Batch, n_features, seq_len)`  
- **LSTM**: `(Batch, seq_len, n_features)`


---
## Definicje modeli


In [59]:
class Conv1DRegressor(nn.Module):
    """
    1D CNN do regresji szeregów czasowych FPL.
    
    Wejście: (B, n_features, seq_len)
    Wyjście: (B, 1)
    
    Architektura:
        - Stos bloków konwolucyjnych z residual skip
        - Global Average Pooling po osi czasu
        - Głowica MLP
    """

    def __init__(self,
                 n_features: int,
                 seq_len: int,
                 channels: list[int] = None,
                 kernel_size: int = 3,
                 dropout: float = 0.2):
        super().__init__()
        if channels is None:
            channels = [64, 128, 64]

        self.input_proj = nn.Conv1d(n_features, channels[0], kernel_size=1)

        blocks = []
        for i in range(len(channels) - 1):
            in_ch  = channels[i]
            out_ch = channels[i + 1]
            blocks.append(self._conv_block(in_ch, out_ch, kernel_size, dropout))
        self.conv_blocks = nn.ModuleList(blocks)

        # Projekcja skip jeśli liczba kanałów się zmienia
        self.skips = nn.ModuleList([
            nn.Conv1d(channels[i], channels[i + 1], kernel_size=1)
            if channels[i] != channels[i + 1] else nn.Identity()
            for i in range(len(channels) - 1)
        ])

        self.gap = nn.AdaptiveAvgPool1d(1)   # (B, C, T) → (B, C, 1)

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels[-1], 32),
            nn.LayerNorm(32),
            nn.LeakyReLU(0.01),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    @staticmethod
    def _conv_block(in_ch: int, out_ch: int, ks: int, dr: float) -> nn.Sequential:
        pad = ks // 2
        return nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=ks, padding=pad),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.Dropout(dr),
            nn.Conv1d(out_ch, out_ch, kernel_size=ks, padding=pad),
            nn.BatchNorm1d(out_ch),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, F, T)
        out = self.input_proj(x)    # (B, C0, T)
        for block, skip in zip(self.conv_blocks, self.skips):
            residual = skip(out)
            out = block(out) + residual
            out = torch.relu(out)
        out = self.gap(out)         # (B, C_last, 1)
        return self.head(out)       # (B, 1)


conv1d_model = Conv1DRegressor(
    n_features=N_FEATURES_SEQ,
    seq_len=SEQ_LEN,
    channels=[128, 64, 32],
    kernel_size=5,
    dropout=0.2
).to(device)

summary(conv1d_model, input_size=(BATCH_SIZE, N_FEATURES_SEQ, SEQ_LEN),
        col_names=["input_size", "output_size", "num_params", "trainable"], col_width=18)


Layer (type:depth-idx)                   Input Shape        Output Shape       Param #            Trainable
Conv1DRegressor                          [512, 32, 4]       [512, 1]           --                 True
├─Conv1d: 1-1                            [512, 32, 4]       [512, 128, 4]      4,224              True
├─ModuleList: 1-2                        --                 --                 --                 True
│    └─Conv1d: 2-1                       [512, 128, 4]      [512, 64, 4]       8,256              True
├─ModuleList: 1-3                        --                 --                 --                 True
│    └─Sequential: 2-2                   [512, 128, 4]      [512, 64, 4]       --                 True
│    │    └─Conv1d: 3-1                  [512, 128, 4]      [512, 64, 4]       41,024             True
│    │    └─BatchNorm1d: 3-2             [512, 64, 4]       [512, 64, 4]       128                True
│    │    └─GELU: 3-3                    [512, 64, 4]       [512, 64

In [60]:
class LSTMRegressor(nn.Module):
    """
    Wielowarstwowy LSTM do regresji punktów FPL.
    
    Wejście: (B, seq_len, n_features)
    Wyjście: (B, 1)
    
    Architektura:
        - Projekcja wejściowa → hidden_size
        - N warstw LSTM z dropout między warstwami
        - Uwaga: używamy tylko ostatniego ukrytego stanu (h_T)
        - Głowica MLP z LayerNorm
    """

    def __init__(self,
                 n_features: int,
                 hidden_size: int = 128,
                 num_layers: int = 2,
                 dropout: float = 0.2,
                 bidirectional: bool = False):
        super().__init__()
        self.hidden_size   = hidden_size
        self.num_layers    = num_layers
        self.bidirectional = bidirectional
        self.directions    = 2 if bidirectional else 1

        self.input_proj = nn.Linear(n_features, hidden_size)

        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )

        lstm_out_size = hidden_size * self.directions

        self.head = nn.Sequential(
            nn.LayerNorm(lstm_out_size),
            nn.Linear(lstm_out_size, 64),
            nn.LeakyReLU(0.01),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F)
        x = self.input_proj(x)          # (B, T, H)
        _, (h_n, _) = self.lstm(x)      # h_n: (num_layers * directions, B, H)

        # Bierzemy ostatnią warstwę (forward + backward jeśli bidi)
        if self.bidirectional:
            h_last = torch.cat([h_n[-2], h_n[-1]], dim=-1)  # (B, 2H)
        else:
            h_last = h_n[-1]                                 # (B, H)

        return self.head(h_last)        # (B, 1)


lstm_model = LSTMRegressor(
    n_features=N_FEATURES_SEQ,
    hidden_size=128,
    num_layers=2,
    dropout=0.2,
    bidirectional=False
).to(device)

summary(lstm_model, input_size=(BATCH_SIZE, SEQ_LEN, N_FEATURES_SEQ),
        col_names=["input_size", "output_size", "num_params", "trainable"], col_width=18)


Layer (type:depth-idx)                   Input Shape        Output Shape       Param #            Trainable
LSTMRegressor                            [512, 4, 32]       [512, 1]           --                 True
├─Linear: 1-1                            [512, 4, 32]       [512, 4, 128]      4,224              True
├─LSTM: 1-2                              [512, 4, 128]      [512, 4, 128]      264,192            True
├─Sequential: 1-3                        [512, 128]         [512, 1]           --                 True
│    └─LayerNorm: 2-1                    [512, 128]         [512, 128]         256                True
│    └─Linear: 2-2                       [512, 128]         [512, 64]          8,256              True
│    └─LeakyReLU: 2-3                    [512, 64]          [512, 64]          --                 --
│    └─Dropout: 2-4                      [512, 64]          [512, 64]          --                 --
│    └─Linear: 2-5                       [512, 64]          [512, 1]    

In [61]:
set_seed(77)

trainer = Trainer(device=device, random_state=77)
loss_fn = HuberLoss(delta=10.0)

NUM_EPOCHS = 300
PATIENCE   = 100

MODEL_REGISTRY = {
    "MLP":    (mlp,          train_dl_mlp,  valid_dl_mlp),
    "Conv1D": (conv1d_model, train_dl_conv, valid_dl_conv),
    "LSTM":   (lstm_model,   train_dl_lstm, valid_dl_lstm),
}

ALL_RESULTS     = {}
ALL_BEST_STATES = {}

for model_name, (model, tr_dl, va_dl) in MODEL_REGISTRY.items():
    print(f"\n{'='*75}", flush=True)
    print(f" Trening: {model_name}", flush=True)
    print(f"{'='*75}\n", flush=True)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )

    results, best_state = trainer.model_eval(
        train_dataloader=tr_dl,
        test_dataloader=va_dl,
        model=model,
        optimizer=optimizer,
        loss_fn=loss_fn,
        num_epochs=NUM_EPOCHS,
        patience=PATIENCE,
    )

    ALL_RESULTS[model_name]     = results
    ALL_BEST_STATES[model_name] = best_state

    pt_path = f"best_{model_name.lower()}.pt"
    torch.save(best_state, pt_path)
    print(f"{model_name} gotowy - epok: {len(results['train_loss'])}", flush=True)

print("\nTrening wszystkich modeli zakonczony.")
print("Rozpoczynam logowanie do MLflow...\n")

client = mlflow.MlflowClient()
import mlflow.entities

for model_name, (model, _, _) in MODEL_REGISTRY.items():
    results    = ALL_RESULTS[model_name]
    best_state = ALL_BEST_STATES[model_name]

    model.load_state_dict(best_state)

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params({
            "model":        model_name,
            "lr":           LR,
            "weight_decay": WEIGHT_DECAY,
            "num_epochs":   NUM_EPOCHS,
            "patience":     PATIENCE,
            "loss_fn":      loss_fn.__class__.__name__,
            "seq_len":      SEQ_LEN if model_name != "MLP" else "n/a",
            "device":       device,
        })

        run_id  = mlflow.active_run().info.run_id
        CHUNK   = 100
        entries = []

        for epoch, (trl, tsl, tsm, tsma, ts_r2) in enumerate(zip(
            results["train_loss"], results["test_loss"],
            results["test_mse"],   results["test_mae"],
            results["test_r2"]
        )):
            ts = int(epoch)
            entries += [
                mlflow.entities.Metric("train_loss", trl,  ts, epoch),
                mlflow.entities.Metric("valid_loss", tsl,  ts, epoch),
                mlflow.entities.Metric("valid_mse",  tsm,  ts, epoch),
                mlflow.entities.Metric("valid_mae",  tsma, ts, epoch),
                mlflow.entities.Metric("valid_r2",   ts_r2, ts, epoch),
            ]

        for i in range(0, len(entries), CHUNK * 5):
            client.log_batch(run_id=run_id, metrics=entries[i:i + CHUNK * 5])

        mlflow.log_metrics({
            "final_valid_loss": results["test_loss"][-1],
            "final_valid_r2":   results["test_r2"][-1],
            "final_valid_mae":  results["test_mae"][-1],
        })

        mlflow.pytorch.log_model(model, artifact_path=f"model_{model_name.lower()}")

        print(f"{model_name} pomyslnie zalogowany do MLflow (Metryki + Model).")

print("\nWszystko gotowe.")


 Trening: MLP



MLP:   0%|          | 0/300 [00:00<?, ?it/s]

Epoch    0 | LR: 1.00e-03
  Train  Loss=0.3833  MSE=0.7661  MAE=0.4889  R2=0.2339
  Valid  Loss=0.3512  MSE=0.7114  MAE=0.4771  R2=0.2805
───────────────────────────────────────────────────────────────────────────
Epoch   30 | LR: 3.13e-05
  Train  Loss=0.3224  MSE=0.6447  MAE=0.4177  R2=0.3553
  Valid  Loss=0.3403  MSE=0.6892  MAE=0.4280  R2=0.3029
───────────────────────────────────────────────────────────────────────────
Epoch   60 | LR: 2.44e-07
  Train  Loss=0.3211  MSE=0.6421  MAE=0.4160  R2=0.3579
  Valid  Loss=0.3406  MSE=0.6898  MAE=0.4252  R2=0.3023
───────────────────────────────────────────────────────────────────────────
Epoch   90 | LR: 1.00e-07
  Train  Loss=0.3212  MSE=0.6425  MAE=0.4165  R2=0.3575
  Valid  Loss=0.3406  MSE=0.6898  MAE=0.4252  R2=0.3023
───────────────────────────────────────────────────────────────────────────
Early stopping po 109 epokach (brak poprawy przez 100 epok)
MLP gotowy - epok: 110

 Trening: Conv1D



Conv1DRegressor:   0%|          | 0/300 [00:00<?, ?it/s]

Epoch    0 | LR: 1.00e-03
  Train  Loss=0.3733  MSE=0.7466  MAE=0.4666  R2=0.2558
  Valid  Loss=0.3524  MSE=0.7165  MAE=0.4838  R2=0.2817
───────────────────────────────────────────────────────────────────────────
Epoch   30 | LR: 1.56e-05
  Train  Loss=0.3240  MSE=0.6479  MAE=0.4223  R2=0.3549
  Valid  Loss=0.3506  MSE=0.7129  MAE=0.4384  R2=0.2854
───────────────────────────────────────────────────────────────────────────
Epoch   60 | LR: 1.00e-07
  Train  Loss=0.3203  MSE=0.6407  MAE=0.4198  R2=0.3620
  Valid  Loss=0.3512  MSE=0.7142  MAE=0.4382  R2=0.2840
───────────────────────────────────────────────────────────────────────────


KeyboardInterrupt: 

In [ ]:
fig, axes = plt.subplots(len(MODEL_REGISTRY), 2, figsize=(14, 4 * len(MODEL_REGISTRY)))

for row, (name, res) in enumerate(ALL_RESULTS.items()):
    rf = pd.DataFrame(res)
    sns.lineplot(data=rf[["train_loss", "test_loss"]], ax=axes[row, 0],
                 palette=["tomato", "steelblue"])
    axes[row, 0].set_title(f"{name} — Loss (Huber)")
    axes[row, 0].set_xlabel("Epoka")

    sns.lineplot(data=rf[["train_r2", "test_r2"]], ax=axes[row, 1],
                 palette=["tomato", "steelblue"])
    axes[row, 1].set_title(f"{name} — R²")
    axes[row, 1].set_xlabel("Epoka")

plt.suptitle("Krzywe uczenia — MLP / Conv1D / LSTM", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()


## Ewaluacja na zbiorze testowym (sezon 25/26)

In [ ]:
TEST_DL_MAP = {
    "MLP":    test_dl_mlp,
    "Conv1D": test_dl_conv,
    "LSTM":   test_dl_lstm,
}

test_metrics = {}

for name, (model, _, _) in MODEL_REGISTRY.items():
    model.load_state_dict(ALL_BEST_STATES[name])
    model.eval()

    test_dl = TEST_DL_MAP[name]
    loss, mse, mae, r2, y_pred_t = trainer.test_step(test_dl, model, loss_fn)

    test_metrics[name] = {
        "loss": loss, "MSE": mse, "MAE": mae, "R2": r2,
        "y_pred": y_pred_t
    }
    print(f"{name:8s} | Loss={loss:.4f}  MSE={mse:.4f}  MAE={mae:.4f}  R2={r2:.4f}")

print("\nPorównanie modeli na teście:")
pd.DataFrame(test_metrics).T[["loss", "MSE", "MAE", "R2"]].round(4)


## Predykcje vs rzeczywiste — MLP (przykład)

In [ ]:
y_test_index = data_splits["y_test"].index

def build_preds_df(y_pred_tensor: torch.Tensor,
                   y_test_index,
                   scaler: StandardScaler,
                   data: pd.DataFrame,
                   model_name: str) -> pd.DataFrame:
    y_pred_inv = scaler.inverse_transform(y_pred_tensor.numpy())
    y_true_inv = scaler.inverse_transform(
        data_splits["y_test"].values
    )
    return pd.DataFrame({
        "model":   model_name,
        "name":    data.loc[y_test_index, "name"].values,
        "season":  data.loc[y_test_index, "season"].values,
        "gw":      data.loc[y_test_index, "gw"].values,
        "y_true":  y_true_inv.flatten(),
        "y_pred":  y_pred_inv.flatten(),
    })

preds_mlp = build_preds_df(
    test_metrics["MLP"]["y_pred"],
    y_test_index, target_scaler, data, "Conw1D"
)
preds_mlp.head(10)


In [ ]:
preds_mlp.y_pred.max(), preds_mlp.y_true.max()

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(12, 5))

sns.histplot(data=preds_mlp, x="y_true", ax=ax[0], bins=30, color="steelblue", label="Rzeczywiste")
sns.histplot(data=preds_mlp, x="y_pred", ax=ax[0], bins=30, color="tomato", alpha=0.6, label="Predykcje MLP")
ax[0].set_title("Rozkład punktów — MLP")
ax[0].legend()

players = preds_mlp["name"].sample(1, random_state=777).values[0]
player_data = preds_mlp[preds_mlp["name"] == players].reset_index(drop=True)
sns.scatterplot(data=player_data, x=player_data.index, y="y_true",
                ax=ax[1], label="Rzeczywiste", color="steelblue")
sns.scatterplot(data=player_data, x=player_data.index, y="y_pred",
                ax=ax[1], label="MLP pred", color="tomato")
ax[1].set_title(f"Predykcje vs rzeczywiste — {players}")
plt.tight_layout()
plt.show()


In [ ]:
"""preds_mlp.to_parquet(config.PREDICTED_STATS_PATH)
print("Predykcje zapisane.")
"""